# RQ3_5 — Interpretation, Robustness & Validation

## Purpose

This notebook is the validation layer for RQ3. It **does not replace RQ3_2, RQ3_3 or RQ3_4**.

The original RQ3 model established a leakage-controlled classification framework using five features confirmed to be present in both PCAOB severity classes:

- Auditing Standard
- Inspection Type
- Country
- Global Network
- Inspection Year

This notebook strengthens that analysis through:

1. Leakage and class-balance validation
2. Leakage-safe preprocessing fitted **only on training data**
3. Comparison of complete-case and missing-category handling
4. Balanced classification metrics, including PR-AUC
5. Sensitivity analysis excluding Inspection Year
6. Temporal holdout validation using the latest inspection year
7. Grouped SHAP interpretation by original feature
8. A final report-ready evidence matrix and methodological decision rule

> **Interpretive boundary:** The classifier identifies predictive associations with PCAOB deficiency severity. It does not establish that any auditing standard, country, network, inspection type, or inspection year causes deficiency severity.


In [14]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

from statsmodels.stats.contingency_tables import mcnemar

from pathlib import Path

DATA_FILE = "pcaob_deficiencies_raw.csv"

REAL_FEATURES = [
    "Auditing Standard",
    "Inspection Type",
    "Country",
    "Global Network",
    "Inspection Year"
]

LEAKAGE_FIELDS = [
    "Audit Area",
    "Finding Count",
    "Paragraph of the Auditing Standard",
    "Firm-Identified Risk Assessment",
    "Issuer Reference Key",
    "Firm played a role but was not the lead auditor",
    "Audits Affected by the Deficiencies Identified in Part I.A",
    "Classification of Audits with Part I.A Deficiencies",
    "Description in the Firm's Inspection Report"
]

TARGET = "severity"

print("RQ3_5 FINAL robustness notebook loaded.")


RQ3_5 FINAL robustness notebook loaded.


## 1. Dataset and leakage validation

The original RQ3 work identified an initial leakage problem in which fields populated only for Part I.A could allow the model to recover the target from feature presence.

The robustness notebook therefore checks:

- the retained five features
- the excluded leakage/structural fields
- class balance
- feature population rates by severity class

A large population-rate difference is treated as a **diagnostic warning**, not automatic proof of leakage.


In [15]:
df = pd.read_csv(DATA_FILE).copy()

# Match the cleaning logic used in the original RQ3 work.
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].replace("", np.nan)

df = df.dropna(how="all").copy()

missing_required = [c for c in REAL_FEATURES + [TARGET] if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")

print(f"Full dataset N = {len(df):,}")
print("\nSeverity distribution:")
display(
    df[TARGET]
    .value_counts()
    .sort_index()
    .rename(index={0: "Part I.B", 1: "Part I.A"})
    .rename("N")
    .to_frame()
)

print("\nSeverity proportions:")
display(
    (df[TARGET].value_counts(normalize=True)
       .sort_index()
       .rename(index={0: "Part I.B", 1: "Part I.A"})
       .mul(100)
       .round(2)
       .rename("Percent")
       .to_frame())
)

print("\nFeature population rate by severity:")
population_rows = []

for col in REAL_FEATURES:
    for sev, label in [(0, "Part I.B"), (1, "Part I.A")]:
        rate = df.loc[df[TARGET] == sev, col].notna().mean() * 100
        population_rows.append({
            "Feature": col,
            "Severity": label,
            "Populated %": rate
        })

population = pd.DataFrame(population_rows)
display(
    population.pivot(index="Feature", columns="Severity", values="Populated %")
    .round(2)
)

print("\nExcluded leakage/structural fields present in source:")
display(pd.DataFrame({"Excluded field": LEAKAGE_FIELDS}))


Full dataset N = 17,077

Severity distribution:


,N
severity,
Part I.B,3034
Part I.A,14043



Severity proportions:


,Percent
severity,
Part I.B,17.77
Part I.A,82.23



Feature population rate by severity:


Severity,Part I.A,Part I.B
Feature,,
Auditing Standard,100.00,99.67
Country,100.00,99.67
Global Network,47.95,24.92
Inspection Type,100.00,99.67
Inspection Year,100.00,99.67



Excluded leakage/structural fields present in source:


,Excluded field
0,Audit Area
1,Finding Count
2,Paragraph of the Auditing Standard
3,Firm-Identified Risk Assessment
4,Issuer Reference Key
5,Firm played a role but was not the lead auditor
6,Audits Affected by the Deficiencies Identified...
7,Classification of Audits with Part I.A Deficie...
8,Description in the Firm's Inspection Report


## 2. Missingness handling as a robustness issue

`Global Network` contains substantial missingness in the supplied PCAOB dataset.

The original RQ3_2 model uses complete-case analysis after dropping rows with missing values in the five predictors.

For robustness, this notebook compares:

- **Complete-case analysis:** reproduces the original cleaning philosophy.
- **Missing-value imputation analysis:** treats missing categorical values as an explicit `"Missing"` category.

The second approach avoids discarding a large portion of the source data and tests whether conclusions depend on complete-case selection.


In [16]:
complete_case = df.dropna(subset=REAL_FEATURES + [TARGET]).copy()

print(f"Complete-case N = {len(complete_case):,}")
print("Complete-case severity distribution:")
display(
    complete_case[TARGET]
    .value_counts()
    .sort_index()
    .rename(index={0: "Part I.B", 1: "Part I.A"})
    .rename("N")
    .to_frame()
)

print("\nMissingness by retained feature:")
missing_summary = pd.DataFrame({
    "Missing N": df[REAL_FEATURES].isna().sum(),
    "Missing %": df[REAL_FEATURES].isna().mean() * 100
}).sort_values("Missing %", ascending=False)

display(missing_summary.round(2))


Complete-case N = 7,489
Complete-case severity distribution:


,N
severity,
Part I.B,756
Part I.A,6733



Missingness by retained feature:


,Missing N,Missing %
Global Network,9588,56.15
Auditing Standard,10,0.06
Inspection Type,10,0.06
Country,10,0.06
Inspection Year,10,0.06


## 3. Leakage-safe preprocessing

The original RQ3_2 encoder is fitted before the train/test split.

For this robustness analysis, preprocessing is deliberately moved **inside a scikit-learn Pipeline**, so the OneHotEncoder and imputers learn their categories/statistics from the training set only.

This prevents test-set preprocessing information from entering model fitting.


In [17]:
def make_preprocessor(features):
    categorical_features = features

    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    ))
                ]),
                categorical_features
            )
        ],
        remainder="drop"
    )


def make_models(features):
    preprocessor = make_preprocessor(features)

    rf = Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    lr = Pipeline([
        ("preprocess", make_preprocessor(features)),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    return rf, lr


def evaluate_model(name, model, X_test, y_test):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba)
    }, pred, proba


## 4. Primary robustness model — leakage-safe random split

The random split retains the original RQ3 evaluation design (`80/20`, stratified, random state 42), but preprocessing is fitted only on the training data.

Because Part I.A is the majority class, **balanced accuracy, F1, ROC-AUC and PR-AUC are emphasized alongside accuracy**.

The majority-class baseline is reported for context.


In [18]:
X = df[REAL_FEATURES].copy()
y = df[TARGET].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf, lr = make_models(REAL_FEATURES)

rf.fit(X_train, y_train)
lr.fit(X_train, y_train)

rf_metrics, rf_pred, rf_proba = evaluate_model(
    "Random Forest", rf, X_test, y_test
)

lr_metrics, lr_pred, lr_proba = evaluate_model(
    "Logistic Regression", lr, X_test, y_test
)

majority_class = y_train.mode()[0]
majority_pred = np.full(len(y_test), majority_class)

baseline = {
    "Model": "Majority baseline",
    "Accuracy": accuracy_score(y_test, majority_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, majority_pred),
    "Precision": precision_score(y_test, majority_pred, zero_division=0),
    "Recall": recall_score(y_test, majority_pred, zero_division=0),
    "F1": f1_score(y_test, majority_pred, zero_division=0),
    "ROC-AUC": np.nan,
    "PR-AUC": np.nan
}

random_split_results = pd.DataFrame(
    [baseline, rf_metrics, lr_metrics]
)

display(random_split_results.round(4))

print("\nRandom Forest confusion matrix:")
display(
    pd.DataFrame(
        confusion_matrix(y_test, rf_pred),
        index=["Actual Part I.B", "Actual Part I.A"],
        columns=["Predicted Part I.B", "Predicted Part I.A"]
    )
)


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Majority baseline,0.8223,0.5000,0.8223,1.0000,0.9025,NaN,NaN
1,Random Forest,0.9450,0.9297,0.9792,0.9534,0.9661,0.9756,0.9926
2,Logistic Regression,0.9646,0.9526,0.9855,0.9712,0.9783,0.9874,0.9968



Random Forest confusion matrix:


,Predicted Part I.B,Predicted Part I.A
Actual Part I.B,550,57
Actual Part I.A,131,2678


## 5. Random Forest vs Logistic Regression — paired comparison

McNemar's test evaluates whether the two classifiers make significantly different paired classification decisions on the same held-out records.

This is a **model comparison**, not evidence that one feature causes severity.


In [19]:
# ============================================================
# 5. RANDOM FOREST VS LOGISTIC REGRESSION — MCNEMAR TEST
# ============================================================

both_correct = np.sum(
    (rf_pred == y_test.to_numpy()) &
    (lr_pred == y_test.to_numpy())
)

rf_only = np.sum(
    (rf_pred == y_test.to_numpy()) &
    (lr_pred != y_test.to_numpy())
)

lr_only = np.sum(
    (rf_pred != y_test.to_numpy()) &
    (lr_pred == y_test.to_numpy())
)

both_wrong = np.sum(
    (rf_pred != y_test.to_numpy()) &
    (lr_pred != y_test.to_numpy())
)

mcnemar_table = [
    [both_correct, rf_only],
    [lr_only, both_wrong]
]

mcnemar_result = mcnemar(
    mcnemar_table,
    exact=False,
    correction=True
)

print("McNemar contingency table:")

display(
    pd.DataFrame(
        mcnemar_table,
        index=["RF correct", "RF wrong"],
        columns=["LogReg correct", "LogReg wrong"]
    )
)

# Avoid reporting an underflowed numerical p-value as exactly zero.
p_mcnemar = mcnemar_result.pvalue

if p_mcnemar < 0.001:
    p_mcnemar_display = "< 0.001"
else:
    p_mcnemar_display = f"= {p_mcnemar:.3f}"

print(
    f"McNemar chi-square = {mcnemar_result.statistic:.4f}; "
    f"p {p_mcnemar_display}"
)

McNemar contingency table:


,LogReg correct,LogReg wrong
RF correct,3208,20
RF wrong,87,101


McNemar chi-square = 40.7103; p < 0.001


## 6. Inspection Year sensitivity

Inspection Year is a legitimate retained predictor because it is populated for both severity classes, but it may encode temporal differences.

Therefore, the Random Forest is compared:

- **Full model:** all five retained features
- **No-year model:** all retained features except Inspection Year

If performance falls substantially after removing year, the predictive model should be described as having a meaningful temporal component.


In [20]:
NO_YEAR_FEATURES = [
    "Auditing Standard",
    "Inspection Type",
    "Country",
    "Global Network"
]

rf_no_year, lr_no_year = make_models(NO_YEAR_FEATURES)

X_no_year = df[NO_YEAR_FEATURES].copy()

Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    X_no_year,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf_no_year.fit(Xn_train, yn_train)

no_year_metrics, no_year_pred, no_year_proba = evaluate_model(
    "Random Forest — no Inspection Year",
    rf_no_year,
    Xn_test,
    yn_test
)

year_sensitivity = pd.DataFrame([
    rf_metrics,
    no_year_metrics
])

display(year_sensitivity.round(4))


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Random Forest,0.9450,0.9297,0.9792,0.9534,0.9661,0.9756,0.9926
1,Random Forest — no Inspection Year,0.9456,0.9294,0.9788,0.9544,0.9665,0.9724,0.9911


## 7. Temporal holdout validation

Random train/test splitting is useful for the primary classification comparison, but Inspection Year makes a temporal validation especially informative.

The latest inspection year in the supplied dataset is used as a future-style holdout:

- **Training:** all earlier inspection years
- **Testing:** latest inspection year

This evaluates whether the model generalizes to a later inspection period rather than merely to randomly selected records from the same years.


In [21]:
year_series = pd.to_numeric(df["Inspection Year"], errors="coerce")
valid_year_mask = year_series.notna()

temporal_df = df.loc[valid_year_mask].copy()
temporal_df["_year_numeric"] = year_series.loc[valid_year_mask].astype(int)

latest_year = temporal_df["_year_numeric"].max()

temporal_train = temporal_df[
    temporal_df["_year_numeric"] < latest_year
].copy()

temporal_test = temporal_df[
    temporal_df["_year_numeric"] == latest_year
].copy()

print(f"Temporal training years: {sorted(temporal_train['_year_numeric'].unique())}")
print(f"Temporal test year: {latest_year}")
print(f"Temporal train N: {len(temporal_train):,}")
print(f"Temporal test N: {len(temporal_test):,}")

X_temporal_train = temporal_train[REAL_FEATURES].copy()
y_temporal_train = temporal_train[TARGET].astype(int)

X_temporal_test = temporal_test[REAL_FEATURES].copy()
y_temporal_test = temporal_test[TARGET].astype(int)

rf_temporal, _ = make_models(REAL_FEATURES)
rf_temporal.fit(X_temporal_train, y_temporal_train)

temporal_metrics, temporal_pred, temporal_proba = evaluate_model(
    f"Random Forest — temporal holdout ({latest_year})",
    rf_temporal,
    X_temporal_test,
    y_temporal_test
)

display(pd.DataFrame([temporal_metrics]).round(4))

print("\nTemporal holdout class distribution:")
display(
    y_temporal_test.value_counts()
    .sort_index()
    .rename(index={0:"Part I.B", 1:"Part I.A"})
    .rename("N")
    .to_frame()
)


Temporal training years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Temporal test year: 2025
Temporal train N: 16,286
Temporal test N: 781


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Random Forest — temporal holdout (2025),0.9129,0.8841,0.9322,0.9487,0.9404,0.9309,0.9516



Temporal holdout class distribution:


,N
severity,
Part I.B,216
Part I.A,565


## 8. Complete-case vs missing-value imputation sensitivity

The original model drops observations with missing retained predictors.

The leakage-safe model above instead uses imputation for categorical predictors.

This section explicitly compares the two analytical approaches so the final report can state whether the conclusions depend on complete-case selection.


In [22]:
# Complete-case random split
cc = complete_case.copy()

X_cc = cc[REAL_FEATURES].copy()
y_cc = cc[TARGET].astype(int)

Xcc_train, Xcc_test, ycc_train, ycc_test = train_test_split(
    X_cc,
    y_cc,
    test_size=0.20,
    random_state=42,
    stratify=y_cc
)

rf_cc, _ = make_models(REAL_FEATURES)
rf_cc.fit(Xcc_train, ycc_train)

cc_metrics, cc_pred, cc_proba = evaluate_model(
    "Random Forest — complete case",
    rf_cc,
    Xcc_test,
    ycc_test
)

# Full dataset with leakage-safe imputation
full_metrics = rf_metrics.copy()
full_metrics["Model"] = "Random Forest — imputation / full data"

comparison = pd.DataFrame([
    full_metrics,
    cc_metrics,
    temporal_metrics
])

display(
    comparison[
        [
            "Model",
            "Accuracy",
            "Balanced Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC-AUC",
            "PR-AUC"
        ]
    ].round(4)
)


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Random Forest — imputation / full data,0.9450,0.9297,0.9792,0.9534,0.9661,0.9756,0.9926
1,Random Forest — complete case,0.9626,0.9410,0.9901,0.9681,0.9790,0.9824,0.9970
2,Random Forest — temporal holdout (2025),0.9129,0.8841,0.9322,0.9487,0.9404,0.9309,0.9516


## 9. Grouped SHAP interpretation

The original RQ3_3 SHAP analysis correctly handles binary-class SHAP output and groups one-hot features back to their original variables.

This robustness section repeats that idea using the **leakage-safe Random Forest** fitted on the training data only.

The key interpretive output is grouped mean absolute SHAP importance for:

- Auditing Standard
- Inspection Type
- Country
- Global Network
- Inspection Year

> A high SHAP value means the feature contributes strongly to the model's prediction. It does **not** establish causation.


In [23]:
import shap

# Use a bounded held-out sample to keep computation practical.
sample_n = min(500, len(X_test))
X_shap = X_test.iloc[:sample_n].copy()

# Extract the fitted preprocessing and RF model from the pipeline.
preprocessor = rf.named_steps["preprocess"]
rf_estimator = rf.named_steps["model"]

X_shap_encoded = preprocessor.transform(X_shap)

feature_names = preprocessor.get_feature_names_out()

explainer = shap.TreeExplainer(rf_estimator)
shap_raw = explainer.shap_values(X_shap_encoded)

if isinstance(shap_raw, list):
    shap_values = np.asarray(shap_raw[1])
elif isinstance(shap_raw, np.ndarray) and shap_raw.ndim == 3:
    shap_values = shap_raw[:, :, 1]
else:
    shap_values = np.asarray(shap_raw)

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    "Encoded Feature": feature_names,
    "Mean Absolute SHAP": mean_abs_shap
}).sort_values(
    "Mean Absolute SHAP",
    ascending=False
)

def original_feature(encoded_name):
    clean_name = encoded_name.replace("categorical__", "")
    for feature in REAL_FEATURES:
        if clean_name.startswith(feature + "_") or clean_name == feature:
            return feature
    return clean_name

shap_df["Original Feature"] = shap_df["Encoded Feature"].apply(
    original_feature
)

grouped_shap = (
    shap_df.groupby("Original Feature")["Mean Absolute SHAP"]
    .sum()
    .sort_values(ascending=False)
)

print("Top 15 encoded features:")
display(shap_df.head(15).round(6))

print("\nGrouped SHAP importance:")
display(grouped_shap.rename("Mean Absolute SHAP").to_frame().round(6))


Top 15 encoded features:


,Encoded Feature,Mean Absolute SHAP,Original Feature
13,categorical__Auditing Standard_AS 2201,0.072603,Auditing Standard
14,categorical__Auditing Standard_AS 2301,0.064174,Auditing Standard
8,categorical__Auditing Standard_AS 1301,0.044574,Auditing Standard
48,categorical__Inspection Type_Annually Inspected,0.037813,Inspection Type
36,categorical__Auditing Standard_AS 3101,0.036505,Auditing Standard
49,categorical__Inspection Type_Triennially Inspe...,0.036291,Inspection Type
2,categorical__Auditing Standard_AS 1105,0.029405,Auditing Standard
17,categorical__Auditing Standard_AS 2315,0.028571,Auditing Standard
43,categorical__Auditing Standard_PCAOB Rule 3211,0.023204,Auditing Standard
22,categorical__Auditing Standard_AS 2501,0.021128,Auditing Standard



Grouped SHAP importance:


,Mean Absolute SHAP
Original Feature,
Auditing Standard,0.370015
Inspection Type,0.074104
Inspection Year,0.019612
Global Network,0.017601
Country,0.005852


## 9b. Caveat: possible definitional circularity, not just missingness leakage

The leakage checks in Section 1 (and in RQ3_2's original checks) only test
whether a feature's **population rate** differs sharply between the two
severity classes. They do not test whether a feature is **quasi-definitional**
to PCAOB's own classification rubric.

`Auditing Standard` is the dominant predictor in every version of this
analysis (grouped SHAP importance well above the next feature, consistently
across RQ3_3, RQ3_3_v2, and this notebook's leakage-safe re-fit). It is
plausible that PCAOB's own internal methodology preferentially routes
findings citing certain auditing standards (e.g. AS 2201, internal control
over financial reporting) into Part I.A rather than Part I.B as a matter of
classification procedure, not because the standard cited is an independently
informative signal about audit risk. If so, the model is partly
reconstructing PCAOB's own classification logic rather than predicting
severity from a genuinely separate signal.

This does not invalidate the result -- the leakage-safe pipeline in this
notebook is methodologically sound for what it tests -- but the unusually
strong performance (F1, ROC-AUC, PR-AUC all high even under the stricter
train-only preprocessing and temporal holdout) should be discussed in the
report with this caveat, rather than presented as evidence of a strong,
independent predictive relationship.

A useful follow-up (not yet run in this notebook): refit the leakage-safe
Random Forest with `Auditing Standard` excluded and report the resulting
drop in F1/ROC-AUC/PR-AUC, to make this caveat concrete rather than
speculative -- the same way Section 6 already does for `Inspection Year`.


## 10. Final RQ3 evidence matrix

The final evidence hierarchy is:

1. **Primary predictive evidence:** leakage-safe Random Forest and Logistic Regression on a stratified random holdout.
2. **Class-imbalance evidence:** balanced accuracy, F1 and PR-AUC rather than accuracy alone.
3. **Preprocessing robustness:** train-only preprocessing and comparison with complete-case analysis.
4. **Temporal robustness:** latest-year holdout.
5. **Year sensitivity:** Random Forest with and without Inspection Year.
6. **Interpretability:** grouped SHAP importance.

The results should not be described as causal determinants of deficiency severity.


In [25]:
# ============================================================
# 10. FINAL RQ3 EVIDENCE MATRIX
# ============================================================

evidence_rows = [

    # --------------------------------------------------------
    # 1. Primary predictive evidence — Random Forest
    # --------------------------------------------------------
    {
        "Evidence": "Leakage-safe Random Forest",
        "Estimate": rf_metrics["F1"],
        "Metric": "F1",
        "p-value": np.nan,
        "Supported?": True,
        "Interpretation": (
            "Strong predictive performance after fitting all "
            "preprocessing steps only on the training data."
        )
    },

    # --------------------------------------------------------
    # 2. PR-AUC
    # --------------------------------------------------------
    {
        "Evidence": "Leakage-safe Random Forest",
        "Estimate": rf_metrics["PR-AUC"],
        "Metric": "PR-AUC",
        "p-value": np.nan,
        "Supported?": True,
        "Interpretation": (
            "High PR-AUC indicates strong discrimination while "
            "accounting for the substantial class imbalance."
        )
    },

    # --------------------------------------------------------
    # 3. Balanced Accuracy
    # --------------------------------------------------------
    {
        "Evidence": "Leakage-safe Random Forest",
        "Estimate": rf_metrics["Balanced Accuracy"],
        "Metric": "Balanced Accuracy",
        "p-value": np.nan,
        "Supported?": True,
        "Interpretation": (
            "Performance remains strong when both severity classes "
            "are weighted equally."
        )
    },

    # --------------------------------------------------------
    # 4. Inspection Year sensitivity
    # --------------------------------------------------------
    {
        "Evidence": "Random Forest without Inspection Year",
        "Estimate": no_year_metrics["F1"],
        "Metric": "F1",
        "p-value": np.nan,
        "Supported?": True,
        "Interpretation": (
            "Removing Inspection Year produces essentially unchanged "
            "F1 performance, indicating that predictive performance "
            "is not materially dependent on Inspection Year."
        )
    },

    # --------------------------------------------------------
    # 5. Temporal holdout
    # --------------------------------------------------------
    {
        "Evidence": f"Temporal holdout ({latest_year})",
        "Estimate": temporal_metrics["F1"],
        "Metric": "F1",
        "p-value": np.nan,
        "Supported?": True,
        "Interpretation": (
            "Performance remains strong on the later inspection-year "
            "holdout, although it is lower than the random-split "
            "performance, indicating some reduction in temporal "
            "generalization."
        )
    },

    # --------------------------------------------------------
    # 6. Complete-case sensitivity
    # --------------------------------------------------------
    {
        "Evidence": "Complete-case Random Forest",
        "Estimate": cc_metrics["F1"],
        "Metric": "F1",
        "p-value": np.nan,
        "Supported?": True,
        "Interpretation": (
            "The complete-case analysis provides a sensitivity check "
            "against the full-data imputation approach."
        )
    },

    # --------------------------------------------------------
    # 7. Random Forest vs Logistic Regression
    # --------------------------------------------------------
    {
        "Evidence": "Random Forest vs Logistic Regression",
        "Estimate": mcnemar_result.statistic,
        "Metric": "McNemar χ²",
        "p-value": mcnemar_result.pvalue,
        "Supported?": np.nan,
        "Interpretation": (
            "The two classifiers produce statistically different "
            "paired error patterns on the same held-out observations."
            if mcnemar_result.pvalue < 0.05
            else
            "No statistically significant difference was detected "
            "in the paired error patterns of the two classifiers."
        )
    }
]

evidence_matrix = pd.DataFrame(evidence_rows)

# Display the final report-ready evidence matrix

evidence_display = evidence_matrix.copy()

evidence_display["p-value"] = evidence_display["p-value"].apply(
    lambda x: "< 0.001" if pd.notna(x) and x < 0.001
    else (f"{x:.3f}" if pd.notna(x) else "—")
)

display(evidence_display.round(5))

,Evidence,Estimate,Metric,p-value,Supported?,Interpretation
0,Leakage-safe Random Forest,0.96609,F1,—,True,Strong predictive performance after fitting al...
1,Leakage-safe Random Forest,0.99264,PR-AUC,—,True,High PR-AUC indicates strong discrimination wh...
2,Leakage-safe Random Forest,0.92973,Balanced Accuracy,—,True,Performance remains strong when both severity ...
3,Random Forest without Inspection Year,0.96647,F1,—,True,Removing Inspection Year produces essentially ...
4,Temporal holdout (2025),0.94035,F1,—,True,Performance remains strong on the later inspec...
5,Complete-case Random Forest,0.97898,F1,—,True,The complete-case analysis provides a sensitiv...
6,Random Forest vs Logistic Regression,40.71028,McNemar χ²,< 0.001,NaN,The two classifiers produce statistically diff...


## 11. Report-ready interpretation

### Recommended interpretation logic

Use the following hierarchy when writing RQ3:

1. **Leakage control comes first.** The original 100% accuracy result is not used as evidence because the earlier feature set contained variables that were structurally or substantively linked to Part I.A severity.
2. **Primary predictive evidence:** evaluate the leakage-safe Random Forest using F1, balanced accuracy, ROC-AUC and PR-AUC rather than accuracy alone.
3. **Preprocessing robustness:** preprocessing is fitted only on training data, preventing test-set category information from entering model fitting.
4. **Inspection Year sensitivity:** compare the full model with a model excluding Inspection Year.
5. **Temporal robustness:** evaluate the model on the latest inspection year as a future-style holdout.
6. **Interpretability:** use grouped SHAP values to identify which retained characteristics contribute most strongly to predictions.

### Recommended wording

> **After removal of leakage-prone variables, the RQ3 classifier uses Auditing Standard, Inspection Type, Country, Global Network, and Inspection Year to distinguish Part I.A and Part I.B PCAOB deficiencies. The leakage-safe Random Forest demonstrates strong predictive performance on the stratified holdout when evaluated using imbalance-aware metrics. Removing Inspection Year produces essentially unchanged predictive performance, indicating that the model is not materially dependent on inspection year. Temporal validation on the latest inspection year results in some reduction relative to the random split but retains strong predictive discrimination, providing evidence of temporal generalization with some performance degradation.**

> **SHAP analysis identifies the relative contribution of the retained characteristics to model predictions, with Auditing Standard providing the dominant grouped contribution. These findings represent predictive associations and model contributions rather than causal determinants of PCAOB deficiency severity.**

### Do not write

- "The model proves that AS 2201 causes severe deficiencies."
- "Country causes PCAOB deficiency severity."
- "Inspection Year causes more severe findings."
- "AI caused the change in PCAOB deficiencies."
- "100% accuracy proves the model is perfect."
- "The Random Forest proves the causes of PCAOB deficiencies."

### Preferred terminology

Use:

- "predictive association"
- "model contribution"
- "deficiency severity classification"
- "leakage-controlled model"
- "imbalance-aware evaluation"
- "temporal generalization"
- "inspection-year sensitivity"
- "SHAP contribution"
- "predictive evidence"
- "does not establish causality"

## 12. Methodological decision rule

### Leakage rule

Only the five retained features confirmed to exist across both severity classes are used for the main RQ3 analysis.

Any feature that is populated only for Part I.A, absent from Part I.B, or directly encodes the severity classification must remain excluded.

### Evaluation rule

Accuracy alone is insufficient because Part I.A is the majority class.

The preferred evidence is:

- Balanced Accuracy
- F1
- PR-AUC
- ROC-AUC
- comparison against the majority baseline

### Preprocessing rule

The preferred robustness model fits imputation and one-hot encoding **only on training data**.

### Temporal rule

If temporal holdout performance is materially lower than random-split performance, report that the model's generalization is sensitive to inspection period.

### Inspection Year rule

If performance changes materially when Inspection Year is removed, report that temporal information contributes substantially to prediction.

### SHAP rule

SHAP values identify features that contribute to model predictions. They should not be interpreted as causal effects.

### Final RQ3 conclusion rule

> **RQ3 should be presented as a predictive classification finding: after leakage-controlled feature selection, contextual PCAOB characteristics contain information useful for distinguishing Part I.A and Part I.B deficiencies. The strength and generalizability of that predictive relationship should be reported using imbalance-aware metrics, temporal validation, and year-sensitivity analysis. The model and SHAP results do not establish causal determinants of deficiency severity.**
